# Preparacion, calidad e integracion analitica

Esta etapa genera copias Parquet, reportes de calidad y un modelo estrella DuckDB. Los originales de `AR/maize` permanecen intactos. No se imputan valores ni se convierten unidades sin documentacion.

## Decisiones de integracion

El rendimiento tiene etiquetas de campana (`planting_year`) y `harvest_year` entre 1970 y 2023. Las fuentes fechadas cubren 2001-2023 y humedad de suelo comienza en 2003. Las etiquetas de plantacion/cosecha no son fechas diarias completas.

El diccionario `metadata/data_dictionary.json` registra `mm0` como escenario asumido, no como unidad; la unidad de `prec` sigue provisional. `vpd` significa vapor pressure deficit, pero su unidad sigue pendiente. `sos/eos` son dia del ano, por lo que `sos > eos` se representa como `crosses_calendar_year`, sin inventar fechas.

El alcance es CY-Bench v1.10, solo `maize/AR`. Sus notas v1.1/1.2 indican correcciones de discrepancias de `adm_id` entre rendimiento y predictores, asi que se comparte `dim_administrative_unit`. Esta dimension incluye la union de IDs; los atributos de `location` estan disponibles solo para parte de ellos.

Se mantienen `dim_season` y `dim_date` separadas. No se recorta el rendimiento, no se iguala la campana al ano calendario y no se crea un training mart hasta definir fecha de pronostico y cutoff.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

ROOT_DIR = Path.cwd()
if not (ROOT_DIR / "src").is_dir():
    ROOT_DIR = ROOT_DIR.parent
if str(ROOT_DIR / "src") not in sys.path:
    sys.path.insert(0, str(ROOT_DIR / "src"))

from maize_forecasting.data_preparation import prepare_standardized_sources
from maize_forecasting.data_quality import audit_data_quality
from maize_forecasting.star_schema import build_star_schema, query_star

## 1. Copias estandarizadas

Se escriben copias Parquet comprimidas en `data/interim/standardized/`. Solo se normalizan espacios/case de identificadores y se parsea `YYYYMMDD` a DATE. No se imputan, filtran, deduplican ni convierten unidades.

In [ ]:
preparation_manifest = prepare_standardized_sources(overwrite=True)
display(pd.DataFrame(preparation_manifest))

## 2. Data quality por fuente

Se revisan nulos, duplicados en claves candidatas, rangos, ceros, cadencia temporal y cobertura de `adm_id`. El solapamiento con `location` es un diagnostico, no prueba equivalencia geografica.

In [ ]:
quality_report = audit_data_quality()
quality_summary = pd.DataFrame([
    {
        "dataset": name,
        "rows": audit["rows"],
        "distinct_adm_id": audit["distinct_adm_id"],
        "null_cells": sum(audit["missing_values"].values()),
        "duplicate_groups": audit["candidate_key_duplicates"]["duplicate_groups"],
        "date_min": (audit["temporal_coverage"] or {}).get("date_min"),
        "date_max": (audit["temporal_coverage"] or {}).get("date_max"),
    }
    for name, audit in quality_report["datasets"].items()
])
display(quality_summary)
display(pd.DataFrame(quality_report["findings"]))

## 3. Modelo estrella en DuckDB

Se materializan dimensiones de cultivo, fecha, campana, fuente y unidad administrativa. Meteorologia, humedad, NDVI, FPAR, rendimiento y tablas estaticas quedan en hechos separados a su grano nativo. `drainage_class` se modela como codigo categorico con etiquetas aun no disponibles. No se integran las series ambientales con las campanas de rendimiento.

In [ ]:
integration_report = build_star_schema(overwrite=True)
print("Estado de integracion analitica:", integration_report["integration_status"])
print("Estado del training mart:", integration_report["training_mart_status"])
print("Base:", integration_report["database"])
display(pd.DataFrame([
    {"table": table, "rows": rows}
    for table, rows in integration_report["table_counts"].items()
]))

## 4. Calendario de campaña y fase

El puente proyecta el DOY estatico sobre el año inicial de `planting_year`. Los hechos fuente permanecen intactos; las vistas agregan `season_day`, `season_progress` y una fase generica (`early/mid/late`) cuando la fecha cae dentro de una campaña. Las fechas de observacion se usan como proxy de disponibilidad. No se calculan features agregadas.

In [ ]:
season_checks = integration_report["season_alignment"]["checks"]
print("Campañas con calendario:", season_checks["mapped_yield_rows"])
print("Campañas sin crop_calendar:", season_checks["missing_crop_calendar_yield_rows"])
print("Intervalos solapados:", season_checks["overlapping_campaign_intervals"])

cutoff_examples = query_star(
    """
    SELECT admin.adm_id, bridge.season_label, bridge.season_start_date,
           bridge.midseason_cutoff_date, bridge.season_end_date, bridge.mapping_status
    FROM bridge_administrative_season AS bridge
    INNER JOIN dim_administrative_unit AS admin USING (administrative_unit_key)
    ORDER BY admin.adm_id, bridge.season_key
    LIMIT 10
    """
)
display(cutoff_examples)

meteorology_stages = query_star(
    """
    SELECT observation_mapping_status, season_phase, COUNT(*) AS observations
    FROM vw_meteorology_season_stage
    GROUP BY observation_mapping_status, season_phase
    ORDER BY observation_mapping_status, season_phase
    """
)
display(meteorology_stages)

## 5. Consultas iniciales para EDA

El resumen por `harvest_year` es descriptivo y no se usa para emparejar datos ambientales. La cobertura meteorologica se consulta por separado. Los reportes detallados quedan en `reports/data_quality/season_alignment_report.json`. El training mart sigue pendiente de seleccion de variables y agregaciones.

In [ ]:
yield_by_harvest_year = query_star(
    """
    SELECT season.harvest_year, COUNT(*) AS observations,
           AVG(fact.yield) AS mean_yield,
           MIN(fact.yield) AS min_yield, MAX(fact.yield) AS max_yield
    FROM fact_yield AS fact
    INNER JOIN dim_season AS season USING (season_key)
    GROUP BY season.harvest_year
    ORDER BY season.harvest_year
    """
)
display(yield_by_harvest_year)

environmental_coverage = query_star(
    """
    SELECT MIN(calendar.calendar_date) AS first_date,
           MAX(calendar.calendar_date) AS last_date,
           COUNT(DISTINCT fact.date_key) AS observed_dates
    FROM fact_meteorology AS fact
    INNER JOIN dim_date AS calendar USING (date_key)
    """
)
display(environmental_coverage)

cross_year_seasons = query_star(
    "SELECT crosses_calendar_year, COUNT(*) AS locations FROM fact_crop_calendar GROUP BY crosses_calendar_year ORDER BY crosses_calendar_year"
)
display(cross_year_seasons)
display(query_star("SELECT source_dataset, variable_name, unit, unit_status FROM source_variable_metadata ORDER BY source_dataset, variable_name"))

## 6. Limites de esta etapa

La alineacion usa un calendario estatico y deja sin cutoff las campañas sin `crop_calendar`. El cutoff marca fechas observadas hasta el punto medio, pero no hay timestamp de publicacion separado. No se agregaron predictores ni se creo el training mart; esas decisiones quedan para la siguiente etapa.

In [ ]:
print(f"Fuentes copiadas: {len(preparation_manifest)}")
print(f"Fuentes auditadas: {len(quality_report['datasets'])}")
print(f"Hechos materializados: {len(integration_report['facts'])}")
print(f"Filas fact_yield: {integration_report['facts']['fact_yield']:,}")
print(f"Campañas con cutoff: {integration_report['season_alignment']['checks']['mapped_yield_rows']:,}")
print(f"Campañas sin calendario: {integration_report['season_alignment']['checks']['missing_crop_calendar_yield_rows']:,}")
print(f"Referencias dimensionales no resueltas: {sum(integration_report['dimension_reference_unmatched_rows'].values())}")